# Practice 3 · Chest X-ray pneumonia (binary) with a custom Dataset

**Interviewer:** "This notebook classifies chest X-rays as normal or pneumonia, with our own Dataset class. It crashes, and a colleague says the model 'stops learning after a few epochs'. Please walk me through it, tell me what you're checking, and fix it."

There are **8 problems**: 3 crash the code, the rest are silent (the code runs, but learns badly or reports wrong numbers).

**How to work**
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells and fix crash errors one at a time. Read each error message: it tells you where to look.
3. Once it runs, go through it again for the silent bugs.

*Runtime → Change runtime type → T4 GPU.*

### Setup (no bugs in this cell)

In [ ]:
!pip -q install medmnist

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import medmnist

splits = {s: medmnist.PneumoniaMNIST(split=s, download=True, size=28, root="/content") for s in ["train", "val", "test"]}
for s, d in splits.items():
    print(s, d.imgs.shape, d.imgs.dtype, "labels:", d.labels.shape, "pneumonia share:", d.labels.mean().round(2))

## Bug hunt starts here
### Settings

In [ ]:
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

all_imgs = np.concatenate([splits[s].imgs for s in ["train", "val", "test"]])
MEAN, STD = all_imgs.mean(), all_imgs.std()

### Dataset

In [ ]:
class XrayDataset(Dataset):
    def __init__(self, images, labels, train=False):
        self.images = images            # (N, 28, 28) uint8
        self.labels = labels            # (N, 1)  0 = normal, 1 = pneumonia
        self.train = train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i])
        img = (img - MEAN) / STD
        if self.train and torch.rand(1) < 0.5:
            img = torch.flip(img, dims=[-1])
        label = int(self.labels[i, 0])
        return img, label


train_ds = XrayDataset(splits["train"].imgs, splits["train"].labels, train=True)
val_ds = XrayDataset(splits["val"].imgs, splits["val"].labels)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

### Model

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(32 * 7 * 7, 1)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return torch.sigmoid(self.fc(x))


model = SmallCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

### Training and validation

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs).squeeze(1)
            preds = (logits > 0.5).long()
            correct += (preds == labels.long()).sum().item()
    val_acc = correct / len(val_ds)
    lr = optimizer.param_groups[0]["lr"]
    print(f"epoch {epoch:2d}: lr {lr:.0e} | train loss {train_loss:.3f} | val acc {val_acc:.3f}")

### Your notes
> Bug: cell … → why it's wrong → my fix → crash or silent?